# Cats vs. Dogs — Transfer Learning & Fine-Tuning Benchmark

This notebook builds a reproducible deep-learning benchmark for binary image classification using **five ImageNet-pretrained CNN backbones**:

- **VGG16**
- **ResNet50**
- **InceptionV3**
- **MobileNetV2**
- **EfficientNetB0**

For each backbone, the workflow has two stages:

1. **Transfer learning** — freeze the pretrained feature extractor and train a lightweight classifier head.
2. **Fine-tuning** — continue from the trained transfer-learning model, unfreeze only the final backbone layers, keep BatchNormalization layers frozen, lower the learning rate, and train again.

The comparison records test accuracy, loss, trainable-parameter count, training time, classification reports, confusion matrices, and ROC-AUC. The same balanced Cats vs. Dogs split is used for every architecture.

> **Reproducibility note:** run the notebook top-to-bottom in a fresh GPU runtime before publishing final metrics. The code is intentionally structured so all reported results come from one consistent execution.


## 0. Setup

In [ ]:
import os
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import tensorflow_datasets as tfds

from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
)

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

sns.set_style("whitegrid")

print("TensorFlow version:", tf.__version__)
print("GPU available:", len(tf.config.list_physical_devices("GPU")) > 0)


## 1. Loading the Same Dataset as Session 15

Same dataset as last session — **cats vs. dogs** from TensorFlow Datasets — so any difference we see
between "trained from scratch" (Session 15) and "transfer learning / fine-tuning" (this session) is a
fair comparison, not an artifact of different data.

One important difference from Session 15's pipeline: back then we immediately resized every image to a
single `150x150` and normalized to `0-1`, because every version used the same custom CNN. This time,
each pretrained architecture expects its **own** input size (`224x224` for four of them, `299x299` for
InceptionV3) and its **own** specific preprocessing (each one was trained with a particular pixel
normalization baked into its very first layer's expectations). So this time we keep the images at their
native resolution for now, and resize/preprocess them differently for each architecture, later.

In [ ]:
raw_ds, ds_info = tfds.load("cats_vs_dogs", split="train", with_info=True, as_supervised=True)

print(ds_info.description[:400], "...")
print("\nTotal images available in TFDS:", ds_info.splits["train"].num_examples)
print("Label names:", ds_info.features["label"].names)

Same `tf.data` chaining approach as Session 15: `.filter()` splits the stream into cats-only and
dogs-only, `.take()` grabs a fixed number from each, `.concatenate()` glues them back together into one
balanced pool. We deliberately **don't** call `.map()` to resize/normalize yet — that comes later, once
per architecture.

In [ ]:
CLASS_NAMES = ["cat", "dog"]
N_PER_CLASS = 1500          # same 3,000-image balanced pool as Session 15

cats_ds = raw_ds.filter(lambda image, label: label == 0).take(N_PER_CLASS)
dogs_ds = raw_ds.filter(lambda image, label: label == 1).take(N_PER_CLASS)
raw_balanced_ds = cats_ds.concatenate(dogs_ds)

print(f"Balanced pool ready: {N_PER_CLASS} cats + {N_PER_CLASS} dogs = {2 * N_PER_CLASS} images")
print("(kept at native resolution — each architecture below resizes + preprocesses it its own way)")

## 2. A Quick Look at the Data

We already explored this dataset thoroughly in Session 15, so this is a quick sanity check rather than
a full exploration: a few example images (at their original, native resolution — no resizing applied
yet) and a confirmation that the classes are balanced by construction.

In [ ]:
def show_examples_from_ds(ds, n, title):
    fig, axes = plt.subplots(1, n, figsize=(2.4 * n, 2.6))
    for ax, (image, label) in zip(axes, tfds.as_numpy(ds.take(n))):
        ax.imshow(image.astype("uint8"))
        ax.axis("off")
    plt.suptitle(title, fontsize=13, fontweight="bold")
    plt.tight_layout()
    plt.show()

show_examples_from_ds(cats_ds, 5, "Example cats (native resolution, before any resizing)")
show_examples_from_ds(dogs_ds, 5, "Example dogs (native resolution, before any resizing)")

In [ ]:
print(f"cats: {N_PER_CLASS}   dogs: {N_PER_CLASS}   "
      f"(balanced by construction — we took exactly {N_PER_CLASS} of each)")

## 3. Splitting Into Train / Validation / Test

In Session 15 we materialized everything into numpy arrays first and used `sklearn`'s `train_test_split`.
This time, since we want to keep images at native resolution until each architecture's own
resize/preprocess step, we do the split **inside `tf.data`** instead: shuffle the whole balanced pool
once with a fixed seed (`reshuffle_each_iteration=False`, so the split is reproducible), then carve it
into three non-overlapping slices with `.take()` / `.skip()`.

This gives every one of the five architectures the *exact same* train/validation/test images — the only
thing that changes between architectures is how those images get resized and preprocessed.

In [ ]:
TOTAL_IMAGES = 2 * N_PER_CLASS
N_TRAIN = int(0.70 * TOTAL_IMAGES)
N_VAL = int(0.15 * TOTAL_IMAGES)
N_TEST = TOTAL_IMAGES - N_TRAIN - N_VAL

# reshuffle_each_iteration=False -> the same fixed random order every time this dataset is iterated,
# which is what makes the take()/skip() split below reproducible and identical across architectures
shuffled_ds = raw_balanced_ds.shuffle(buffer_size=TOTAL_IMAGES, seed=SEED, reshuffle_each_iteration=False)

train_raw_ds = shuffled_ds.take(N_TRAIN)
val_raw_ds = shuffled_ds.skip(N_TRAIN).take(N_VAL)
test_raw_ds = shuffled_ds.skip(N_TRAIN + N_VAL).take(N_TEST)

print(f"train: {N_TRAIN} images   val: {N_VAL} images   test: {N_TEST} images")

## 4. Shared Settings, Augmentation, Callbacks, and Reporting

The workflow below uses the same training policy for all five backbones:

- **Transfer stage:** frozen ImageNet backbone + a small classification head.
- **Fine-tuning stage:** continue from the trained transfer model, unfreeze only the final backbone block(s), keep `BatchNormalization` layers frozen, and use a much smaller learning rate.
- **Regularization:** light data augmentation, dropout, early stopping, and learning-rate reduction.
- **Evaluation:** accuracy, precision, recall, ROC-AUC, classification report, and confusion matrix.

This makes the comparison more realistic than training each fine-tuned model from a fresh random head.


In [ ]:
BATCH_SIZE = 32
EPOCHS_TL = 8
EPOCHS_FT = 8

results = {}
histories = {}

data_augmentation = keras.Sequential(
    [
        layers.RandomFlip("horizontal", seed=SEED),
        layers.RandomRotation(0.05, seed=SEED),
        layers.RandomZoom(0.10, seed=SEED),
    ],
    name="augmentation",
)


def make_metrics():
    return [
        keras.metrics.BinaryAccuracy(name="accuracy"),
        keras.metrics.Precision(name="precision"),
        keras.metrics.Recall(name="recall"),
        keras.metrics.AUC(name="auc"),
    ]


def make_callbacks():
    return [
        keras.callbacks.EarlyStopping(
            monitor="val_loss",
            patience=2,
            restore_best_weights=True,
            verbose=1,
        ),
        keras.callbacks.ReduceLROnPlateau(
            monitor="val_loss",
            factor=0.2,
            patience=1,
            min_lr=1e-7,
            verbose=1,
        ),
    ]


def count_trainable_params(model):
    return int(np.sum([np.prod(w.shape) for w in model.trainable_weights]))


def unfreeze_last_layers(backbone, n_layers):
    backbone.trainable = True

    for layer in backbone.layers[:-n_layers]:
        layer.trainable = False

    # BatchNorm statistics can become unstable on small datasets / batches.
    for layer in backbone.layers:
        if isinstance(layer, layers.BatchNormalization):
            layer.trainable = False

    return sum(layer.trainable for layer in backbone.layers)


def plot_all_metrics(history, title):
    metric_names = [
        k for k in history.history
        if not k.startswith("val_") and k != "learning_rate"
    ]

    n = len(metric_names)
    fig, axes = plt.subplots(1, n, figsize=(4.2 * n, 4))
    if n == 1:
        axes = [axes]

    for ax, metric in zip(axes, metric_names):
        ax.plot(history.history[metric], label="train", lw=2)
        val_key = f"val_{metric}"

        if val_key in history.history:
            ax.plot(history.history[val_key], label="validation", lw=2)

        ax.set_title(metric)
        ax.set_xlabel("epoch")
        ax.legend()

    plt.suptitle(title, fontsize=14, fontweight="bold")
    plt.tight_layout()
    plt.show()


def evaluate_and_report(model, test_ds, title):
    y_test = np.concatenate([y for _, y in tfds.as_numpy(test_ds)]).astype(int)

    metrics_dict = model.evaluate(test_ds, verbose=0, return_dict=True)

    y_pred_prob = model.predict(test_ds, verbose=0).ravel()
    y_pred = (y_pred_prob >= 0.5).astype(int)

    auc = roc_auc_score(y_test, y_pred_prob)

    print(f"=== {title}: test set ===")
    for name, value in metrics_dict.items():
        print(f"{name}: {value:.4f}")
    print(f"sklearn ROC-AUC: {auc:.4f}\n")

    print(
        classification_report(
            y_test,
            y_pred,
            target_names=CLASS_NAMES,
            digits=3,
        )
    )

    return (
        y_test,
        metrics_dict["loss"],
        metrics_dict["accuracy"],
        y_pred,
        y_pred_prob,
    )


def plot_confusion(y_true, y_pred, title):
    cm = confusion_matrix(y_true, y_pred)

    plt.figure(figsize=(4.5, 4))
    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap="Blues",
        cbar=False,
        xticklabels=CLASS_NAMES,
        yticklabels=CLASS_NAMES,
    )
    plt.title(f"{title} — confusion matrix")
    plt.xlabel("predicted")
    plt.ylabel("actual")
    plt.tight_layout()
    plt.show()


## 5. VGG16

**VGG16** (2014) is the "obvious idea done thoroughly" architecture: nothing but 3x3 convolutions and
2x2 max-pooling, stacked 16 layers deep, with no fancy skip connections or parallel branches. It's
large (~138M parameters, almost all in its fully-connected layers, which we discard here since we only
keep the convolutional base) and slow by modern standards, but its simplicity makes it a good baseline
for "does transfer learning even work here" before trying the more sophisticated architectures.

### 5.1 Resize and preprocess the data for VGG16

In [ ]:
IMG_SIZE_VGG = (224, 224)

def vgg_train_preprocess(image, label):
    image = tf.image.resize(image, IMG_SIZE_VGG)
    image = data_augmentation(image, training=True)
    image = keras.applications.vgg16.preprocess_input(image)
    return image, label


def vgg_eval_preprocess(image, label):
    image = tf.image.resize(image, IMG_SIZE_VGG)
    image = keras.applications.vgg16.preprocess_input(image)
    return image, label


vgg_train_ds = (
    train_raw_ds
    .map(vgg_train_preprocess, num_parallel_calls=tf.data.AUTOTUNE)
    .shuffle(N_TRAIN, seed=SEED)
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)

vgg_val_ds = (
    val_raw_ds
    .map(vgg_eval_preprocess, num_parallel_calls=tf.data.AUTOTUNE)
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)

vgg_test_ds = (
    test_raw_ds
    .map(vgg_eval_preprocess, num_parallel_calls=tf.data.AUTOTUNE)
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)


### 5.2 Model 1 — Transfer learning: bring in the pretrained VGG16 backbone and freeze it

In [ ]:
vgg_tl_base = keras.applications.VGG16(include_top=False , weights="imagenet",
                                       input_shape=(*IMG_SIZE_VGG,3))
vgg_tl_base.trainable = False
print(f"VG16 layers : {len(vgg_tl_base.layers)} , parametes : {vgg_tl_base.count_params():}")

### 5.3 Build the classifier head

A `GlobalAveragePooling2D` head is used instead of `Flatten`. This keeps the trainable head compact and reduces overfitting risk while preserving the pretrained spatial features.


In [ ]:
vgg_tl_model = keras.Sequential(
    [
        vgg_tl_base,
        layers.GlobalAveragePooling2D(),
        layers.Dropout(0.30),
        layers.Dense(1, activation="sigmoid"),
    ],
    name="vgg16_transfer",
)

vgg_tl_model.summary()


### 5.4 Compile — only the head's weights are trainable

In [ ]:
vgg_tl_model.compile(
    optimizer=keras.optimizers.Adam(1e-3),
    loss="binary_crossentropy",
    metrics=make_metrics(),
)

vgg_tl_trainable = count_trainable_params(vgg_tl_model)
print("Trainable parameters:", f"{vgg_tl_trainable:,}")


### 5.5 Train Model 1 (transfer learning)

In [ ]:
t0 = time.time()

vgg_tl_history = vgg_tl_model.fit(
    vgg_train_ds,
    validation_data=vgg_val_ds,
    epochs=EPOCHS_TL,
    callbacks=make_callbacks(),
    verbose=2,
)

vgg_tl_train_time = time.time() - t0


### 5.6 Training curves — every tracked metric, per epoch

In [ ]:

plot_all_metrics(vgg_tl_history, "VGG16 — Transfer Learning")

### 5.7 Evaluate on the test set

In [ ]:
vgg_tl_y_test, vgg_tl_loss, vgg_tl_acc, vgg_tl_y_pred, vgg_tl_y_prob = evaluate_and_report(
    vgg_tl_model,
    vgg_test_ds,
    "VGG16 — Transfer Learning",
)


### 5.8 Confusion matrix

In [ ]:
plot_confusion(vgg_tl_y_test, vgg_tl_y_pred, "VGG16 — Transfer Learning")

In [ ]:
results["VGG16 (Transfer)"] = dict(
    test_loss=vgg_tl_loss,
    test_acc=vgg_tl_acc,
    params=vgg_tl_trainable,
    train_time=vgg_tl_train_time,
)
histories["VGG16 (Transfer)"] = vgg_tl_history


**Takeaway.** VGG16's frozen features come from an enormous, generic dataset (ImageNet) that includes
plenty of cats and dogs among its 1,000 categories — so even *without* any fine-tuning, expect this to
already beat every from-scratch CNN from Session 15 by a wide margin.

### 5.9 Model 2 — Fine-tune the trained VGG16 model

Fine-tuning now starts from the **already-trained transfer-learning checkpoint** instead of a fresh random classifier head. Only the last few convolutional layers are unfrozen and the model is recompiled with a much smaller learning rate.


In [ ]:
vgg_ft_model = vgg_tl_model
vgg_ft_base = vgg_ft_model.layers[0]

n_trainable_layers = unfreeze_last_layers(vgg_ft_base, 4)

print("Trainable backbone layers:", n_trainable_layers)
print("Fine-tuning from the already-trained transfer-learning checkpoint.")


### 5.10 Reuse the trained classifier head

The classifier head keeps the weights learned during the frozen-backbone stage. This is the standard two-stage transfer-learning workflow.


In [ ]:
vgg_ft_model.summary()


### 5.11 Recompile with a smaller learning rate

Changing `.trainable` flags requires recompiling the model. The smaller learning rate protects the pretrained representation while allowing the final layers to adapt.


In [ ]:
vgg_ft_model.compile(
    optimizer=keras.optimizers.Adam(1e-5),
    loss="binary_crossentropy",
    metrics=make_metrics(),
)

vgg_ft_trainable = count_trainable_params(vgg_ft_model)
print("Trainable parameters:", f"{vgg_ft_trainable:,}")


### 5.12 Fine-tune the model

In [ ]:
t0 = time.time()

vgg_ft_history = vgg_ft_model.fit(
    vgg_train_ds,
    validation_data=vgg_val_ds,
    epochs=EPOCHS_FT,
    callbacks=make_callbacks(),
    verbose=2,
)

vgg_ft_train_time = time.time() - t0


### 5.13 Training curves — every tracked metric, per epoch

In [ ]:
plot_all_metrics(vgg_ft_history, "VGG16 — Fine-Tuning")

### 5.14 Evaluate on the test set

In [ ]:
vgg_ft_y_test, vgg_ft_loss, vgg_ft_acc, vgg_ft_y_pred, vgg_ft_y_prob = evaluate_and_report(
    vgg_ft_model,
    vgg_test_ds,
    "VGG16 — Fine-Tuning",
)


### 5.15 Confusion matrix

In [ ]:
plot_confusion(vgg_ft_y_test, vgg_ft_y_pred, "VGG16 — Fine-Tuning")

In [ ]:
results["VGG16 (Fine-tune)"] = dict(
    test_loss=vgg_ft_loss,
    test_acc=vgg_ft_acc,
    params=vgg_ft_trainable,
    train_time=vgg_ft_train_time,
)
histories["VGG16 (Fine-tune)"] = vgg_ft_history


**Takeaway.** Compare the frozen VGG16 baseline with the fine-tuned checkpoint. Fine-tuning should be judged by validation/test improvement, not by training accuracy alone. If validation loss rises while training loss keeps falling, the model is overfitting and early stopping should keep the best checkpoint.


## 6. ResNet50

**ResNet50** (2015) introduced **residual (skip) connections** — each block adds its input back to its
output, letting gradients flow directly through the network during backpropagation. This solved the
"very deep networks are hard to train" problem and let architectures grow far deeper than before
(50 layers here, and much deeper variants exist). It's a much more efficient use of parameters than
VGG16 (~25M vs. ~138M) despite being deeper.

### 6.1 Resize and preprocess the data for ResNet50

In [ ]:
IMG_SIZE_RESNET = (224, 224)

def resnet_train_preprocess(image, label):
    image = tf.image.resize(image, IMG_SIZE_RESNET)
    image = data_augmentation(image, training=True)
    image = keras.applications.resnet50.preprocess_input(image)
    return image, label


def resnet_eval_preprocess(image, label):
    image = tf.image.resize(image, IMG_SIZE_RESNET)
    image = keras.applications.resnet50.preprocess_input(image)
    return image, label


resnet_train_ds = (
    train_raw_ds
    .map(resnet_train_preprocess, num_parallel_calls=tf.data.AUTOTUNE)
    .shuffle(N_TRAIN, seed=SEED)
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)

resnet_val_ds = (
    val_raw_ds
    .map(resnet_eval_preprocess, num_parallel_calls=tf.data.AUTOTUNE)
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)

resnet_test_ds = (
    test_raw_ds
    .map(resnet_eval_preprocess, num_parallel_calls=tf.data.AUTOTUNE)
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)


### 6.2 Model 1 — Transfer learning: bring in the pretrained ResNet50 backbone and freeze it

In [ ]:
resnet_tl_base = keras.applications.ResNet50(
    include_top=False,
    weights="imagenet",
    input_shape=(*IMG_SIZE_RESNET, 3),
)
resnet_tl_base.trainable = False

print("Backbone layers:", len(resnet_tl_base.layers))
print("Backbone parameters:", f"{resnet_tl_base.count_params():,}")


### 6.3 Build the classifier head

The pretrained backbone is followed by `GlobalAveragePooling2D`, dropout, and a sigmoid output. Global average pooling keeps the head compact and helps reduce overfitting compared with flattening the full feature map.


In [ ]:
resnet_tl_model = keras.Sequential(
    [
        resnet_tl_base,
        layers.GlobalAveragePooling2D(),
        layers.Dropout(0.30),
        layers.Dense(1, activation="sigmoid"),
    ],
    name="resnet50_transfer",
)

resnet_tl_model.summary()


### 6.4 Compile — only the head's weights are trainable

In [ ]:
resnet_tl_model.compile(
    optimizer=keras.optimizers.Adam(1e-3),
    loss="binary_crossentropy",
    metrics=make_metrics(),
)

resnet_tl_trainable = count_trainable_params(resnet_tl_model)
print("Trainable parameters:", f"{resnet_tl_trainable:,}")


### 6.5 Train Model 1 (transfer learning)

In [ ]:
t0 = time.time()

resnet_tl_history = resnet_tl_model.fit(
    resnet_train_ds,
    validation_data=resnet_val_ds,
    epochs=EPOCHS_TL,
    callbacks=make_callbacks(),
    verbose=2,
)

resnet_tl_train_time = time.time() - t0


### 6.6 Training curves — every tracked metric, per epoch

In [ ]:
plot_all_metrics(resnet_tl_history, "ResNet50 — Transfer Learning")


### 6.7 Evaluate on the test set

In [ ]:
resnet_tl_y_test, resnet_tl_loss, resnet_tl_acc, resnet_tl_y_pred, resnet_tl_y_prob = evaluate_and_report(
    resnet_tl_model,
    resnet_test_ds,
    "ResNet50 — Transfer Learning",
)


### 6.8 Confusion matrix

In [ ]:
plot_confusion(resnet_tl_y_test, resnet_tl_y_pred, "ResNet50 — Transfer Learning")


In [ ]:
results["ResNet50 (Transfer)"] = dict(
    test_loss=resnet_tl_loss, test_acc=resnet_tl_acc,
    params=resnet_tl_trainable, train_time=resnet_tl_train_time,
)
histories["ResNet50 (Transfer)"] = resnet_tl_history

**Takeaway.** Compare ResNet50's frozen-backbone accuracy and parameter count against VGG16's — a good
architecture often gets you more accuracy *per parameter*, not just more accuracy outright.

### 6.9 Model 2 — Fine-tune the trained ResNet50 model

Continue from the trained transfer-learning checkpoint. Unfreeze the final 35 backbone layers while keeping `BatchNormalization` layers frozen.


In [ ]:
resnet_ft_model = resnet_tl_model
resnet_ft_base = resnet_ft_model.layers[0]

n_trainable_layers = unfreeze_last_layers(resnet_ft_base, 35)

print("Trainable backbone layers:", n_trainable_layers)
print("Fine-tuning from the already-trained transfer-learning checkpoint.")


### 6.10 Reuse the trained classifier head

In [ ]:
resnet_ft_model.summary()


### 6.11 Recompile with a smaller learning rate

In [ ]:
resnet_ft_model.compile(
    optimizer=keras.optimizers.Adam(1e-5),
    loss="binary_crossentropy",
    metrics=make_metrics(),
)

resnet_ft_trainable = count_trainable_params(resnet_ft_model)
print("Trainable parameters:", f"{resnet_ft_trainable:,}")


### 6.12 Fine-tune the model

In [ ]:
t0 = time.time()

resnet_ft_history = resnet_ft_model.fit(
    resnet_train_ds,
    validation_data=resnet_val_ds,
    epochs=EPOCHS_FT,
    callbacks=make_callbacks(),
    verbose=2,
)

resnet_ft_train_time = time.time() - t0


### 6.13 Training curves — every tracked metric, per epoch

In [ ]:
plot_all_metrics(resnet_ft_history, "ResNet50 — Fine-Tuning")


### 6.14 Evaluate on the test set

In [ ]:
resnet_ft_y_test, resnet_ft_loss, resnet_ft_acc, resnet_ft_y_pred, resnet_ft_y_prob = evaluate_and_report(
    resnet_ft_model,
    resnet_test_ds,
    "ResNet50 — Fine-Tuning",
)


### 6.15 Confusion matrix

In [ ]:
plot_confusion(resnet_ft_y_test, resnet_ft_y_pred, "ResNet50 — Fine-Tuning")


In [ ]:
results["ResNet50 (Fine-tune)"] = dict(
    test_loss=resnet_ft_loss, test_acc=resnet_ft_acc,
    params=resnet_ft_trainable, train_time=resnet_ft_train_time,
)
histories["ResNet50 (Fine-tune)"] = resnet_ft_history

**Takeaway.** ResNet50 contains many `BatchNormalization` layers. They remain frozen during fine-tuning so their moving statistics are not destabilized by the relatively small training set and batch size.


## 7. InceptionV3

**InceptionV3** (2015) runs several convolution sizes (1x1, 3x3, 5x5) **in parallel** within each block
and concatenates their outputs, instead of picking one kernel size per layer like VGG16 does — the
network gets to use whichever receptive field works best at each stage. It's also the one architecture
here that expects a larger `299x299` input (versus `224x224` for the rest), which is why its
preprocessing cell below uses a different `IMG_SIZE_INCEPTION` instead of the size the other
architectures use.

### 7.1 Resize and preprocess the data for InceptionV3

In [ ]:
IMG_SIZE_INCEPTION = (299, 299)

def inception_train_preprocess(image, label):
    image = tf.image.resize(image, IMG_SIZE_INCEPTION)
    image = data_augmentation(image, training=True)
    image = keras.applications.inception_v3.preprocess_input(image)
    return image, label


def inception_eval_preprocess(image, label):
    image = tf.image.resize(image, IMG_SIZE_INCEPTION)
    image = keras.applications.inception_v3.preprocess_input(image)
    return image, label


inception_train_ds = (
    train_raw_ds
    .map(inception_train_preprocess, num_parallel_calls=tf.data.AUTOTUNE)
    .shuffle(N_TRAIN, seed=SEED)
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)

inception_val_ds = (
    val_raw_ds
    .map(inception_eval_preprocess, num_parallel_calls=tf.data.AUTOTUNE)
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)

inception_test_ds = (
    test_raw_ds
    .map(inception_eval_preprocess, num_parallel_calls=tf.data.AUTOTUNE)
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)


### 7.2 Model 1 — Transfer learning: bring in the pretrained InceptionV3 backbone and freeze it

In [ ]:
inception_tl_base = keras.applications.InceptionV3(
    include_top=False,
    weights="imagenet",
    input_shape=(*IMG_SIZE_INCEPTION, 3),
)
inception_tl_base.trainable = False

print("Backbone layers:", len(inception_tl_base.layers))
print("Backbone parameters:", f"{inception_tl_base.count_params():,}")


### 7.3 Build the classifier head

The pretrained backbone is followed by `GlobalAveragePooling2D`, dropout, and a sigmoid output. Global average pooling keeps the head compact and helps reduce overfitting compared with flattening the full feature map.


In [ ]:
inception_tl_model = keras.Sequential(
    [
        inception_tl_base,
        layers.GlobalAveragePooling2D(),
        layers.Dropout(0.30),
        layers.Dense(1, activation="sigmoid"),
    ],
    name="inceptionv3_transfer",
)

inception_tl_model.summary()


### 7.4 Compile — only the head's weights are trainable

In [ ]:
inception_tl_model.compile(
    optimizer=keras.optimizers.Adam(1e-3),
    loss="binary_crossentropy",
    metrics=make_metrics(),
)

inception_tl_trainable = count_trainable_params(inception_tl_model)
print("Trainable parameters:", f"{inception_tl_trainable:,}")


### 7.5 Train Model 1 (transfer learning)

In [ ]:
t0 = time.time()

inception_tl_history = inception_tl_model.fit(
    inception_train_ds,
    validation_data=inception_val_ds,
    epochs=EPOCHS_TL,
    callbacks=make_callbacks(),
    verbose=2,
)

inception_tl_train_time = time.time() - t0


### 7.6 Training curves — every tracked metric, per epoch

In [ ]:
plot_all_metrics(inception_tl_history, "InceptionV3 — Transfer Learning")


### 7.7 Evaluate on the test set

In [ ]:
inception_tl_y_test, inception_tl_loss, inception_tl_acc, inception_tl_y_pred, inception_tl_y_prob = evaluate_and_report(
    inception_tl_model,
    inception_test_ds,
    "InceptionV3 — Transfer Learning",
)


### 7.8 Confusion matrix

In [ ]:
plot_confusion(inception_tl_y_test, inception_tl_y_pred, "InceptionV3 — Transfer Learning")


In [ ]:
results["InceptionV3 (Transfer)"] = dict(
    test_loss=inception_tl_loss, test_acc=inception_tl_acc,
    params=inception_tl_trainable, train_time=inception_tl_train_time,
)
histories["InceptionV3 (Transfer)"] = inception_tl_history

**Takeaway.** InceptionV3 sees images at roughly 1.75x the resolution of the other four architectures —
worth keeping in mind if its results differ noticeably; that's a genuinely different amount of visual
detail per image, not just a different backbone.

### 7.9 Model 2 — Fine-tune the trained InceptionV3 model

Continue from the transfer-learning checkpoint and unfreeze the final 60 backbone layers. BatchNormalization layers stay frozen.


In [ ]:
inception_ft_model = inception_tl_model
inception_ft_base = inception_ft_model.layers[0]

n_trainable_layers = unfreeze_last_layers(inception_ft_base, 60)

print("Trainable backbone layers:", n_trainable_layers)
print("Fine-tuning from the already-trained transfer-learning checkpoint.")


### 7.10 Reuse the trained classifier head

In [ ]:
inception_ft_model.summary()


### 7.11 Recompile with a smaller learning rate

In [ ]:
inception_ft_model.compile(
    optimizer=keras.optimizers.Adam(1e-5),
    loss="binary_crossentropy",
    metrics=make_metrics(),
)

inception_ft_trainable = count_trainable_params(inception_ft_model)
print("Trainable parameters:", f"{inception_ft_trainable:,}")


### 7.12 Fine-tune the model

In [ ]:
t0 = time.time()

inception_ft_history = inception_ft_model.fit(
    inception_train_ds,
    validation_data=inception_val_ds,
    epochs=EPOCHS_FT,
    callbacks=make_callbacks(),
    verbose=2,
)

inception_ft_train_time = time.time() - t0


### 7.13 Training curves — every tracked metric, per epoch

In [ ]:
plot_all_metrics(inception_ft_history, "InceptionV3 — Fine-Tuning")


### 7.14 Evaluate on the test set

In [ ]:
inception_ft_y_test, inception_ft_loss, inception_ft_acc, inception_ft_y_pred, inception_ft_y_prob = evaluate_and_report(
    inception_ft_model,
    inception_test_ds,
    "InceptionV3 — Fine-Tuning",
)


### 7.15 Confusion matrix

In [ ]:
plot_confusion(inception_ft_y_test, inception_ft_y_pred, "InceptionV3 — Fine-Tuning")


In [ ]:
results["InceptionV3 (Fine-tune)"] = dict(
    test_loss=inception_ft_loss, test_acc=inception_ft_acc,
    params=inception_ft_trainable, train_time=inception_ft_train_time,
)
histories["InceptionV3 (Fine-tune)"] = inception_ft_history

**Takeaway.** InceptionV3 uses a larger 299×299 input, so its accuracy and compute cost should be interpreted together. Fine-tuning improves the comparison only if it delivers meaningful validation/test gains without excessive extra training cost.


## 8. MobileNetV2

**MobileNetV2** (2018) was designed for phones and other resource-constrained devices, using
**depthwise-separable convolutions** — splitting a standard convolution into a much cheaper "filter each
channel separately, then mix channels" pair of steps. The result is a fraction of the parameters and
compute of the other architectures here (~3.5M vs. tens of millions), aimed at running fast on a device
with no GPU rather than squeezing out the last percent of accuracy.

### 8.1 Resize and preprocess the data for MobileNetV2

In [ ]:
IMG_SIZE_MOBILENET = (224, 224)

def mobilenet_train_preprocess(image, label):
    image = tf.image.resize(image, IMG_SIZE_MOBILENET)
    image = data_augmentation(image, training=True)
    image = keras.applications.mobilenet_v2.preprocess_input(image)
    return image, label


def mobilenet_eval_preprocess(image, label):
    image = tf.image.resize(image, IMG_SIZE_MOBILENET)
    image = keras.applications.mobilenet_v2.preprocess_input(image)
    return image, label


mobilenet_train_ds = (
    train_raw_ds
    .map(mobilenet_train_preprocess, num_parallel_calls=tf.data.AUTOTUNE)
    .shuffle(N_TRAIN, seed=SEED)
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)

mobilenet_val_ds = (
    val_raw_ds
    .map(mobilenet_eval_preprocess, num_parallel_calls=tf.data.AUTOTUNE)
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)

mobilenet_test_ds = (
    test_raw_ds
    .map(mobilenet_eval_preprocess, num_parallel_calls=tf.data.AUTOTUNE)
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)


### 8.2 Model 1 — Transfer learning: bring in the pretrained MobileNetV2 backbone and freeze it

In [ ]:
mobilenet_tl_base = keras.applications.MobileNetV2(
    include_top=False,
    weights="imagenet",
    input_shape=(*IMG_SIZE_MOBILENET, 3),
)
mobilenet_tl_base.trainable = False

print("Backbone layers:", len(mobilenet_tl_base.layers))
print("Backbone parameters:", f"{mobilenet_tl_base.count_params():,}")


### 8.3 Build the classifier head

The pretrained backbone is followed by `GlobalAveragePooling2D`, dropout, and a sigmoid output. Global average pooling keeps the head compact and helps reduce overfitting compared with flattening the full feature map.


In [ ]:
mobilenet_tl_model = keras.Sequential(
    [
        mobilenet_tl_base,
        layers.GlobalAveragePooling2D(),
        layers.Dropout(0.30),
        layers.Dense(1, activation="sigmoid"),
    ],
    name="mobilenetv2_transfer",
)

mobilenet_tl_model.summary()


### 8.4 Compile — only the head's weights are trainable

In [ ]:
mobilenet_tl_model.compile(
    optimizer=keras.optimizers.Adam(1e-3),
    loss="binary_crossentropy",
    metrics=make_metrics(),
)

mobilenet_tl_trainable = count_trainable_params(mobilenet_tl_model)
print("Trainable parameters:", f"{mobilenet_tl_trainable:,}")


### 8.5 Train Model 1 (transfer learning)

In [ ]:
t0 = time.time()

mobilenet_tl_history = mobilenet_tl_model.fit(
    mobilenet_train_ds,
    validation_data=mobilenet_val_ds,
    epochs=EPOCHS_TL,
    callbacks=make_callbacks(),
    verbose=2,
)

mobilenet_tl_train_time = time.time() - t0


### 8.6 Training curves — every tracked metric, per epoch

In [ ]:
plot_all_metrics(mobilenet_tl_history, "MobileNetV2 — Transfer Learning")


### 8.7 Evaluate on the test set

In [ ]:
mobilenet_tl_y_test, mobilenet_tl_loss, mobilenet_tl_acc, mobilenet_tl_y_pred, mobilenet_tl_y_prob = evaluate_and_report(
    mobilenet_tl_model,
    mobilenet_test_ds,
    "MobileNetV2 — Transfer Learning",
)


### 8.8 Confusion matrix

In [ ]:
plot_confusion(mobilenet_tl_y_test, mobilenet_tl_y_pred, "MobileNetV2 — Transfer Learning")


In [ ]:
results["MobileNetV2 (Transfer)"] = dict(
    test_loss=mobilenet_tl_loss, test_acc=mobilenet_tl_acc,
    params=mobilenet_tl_trainable, train_time=mobilenet_tl_train_time,
)
histories["MobileNetV2 (Transfer)"] = mobilenet_tl_history

**Takeaway.** This is the efficiency question in miniature: does MobileNetV2 give up much accuracy
compared to the much larger VGG16/ResNet50/InceptionV3, or does it hold its own with a fraction of the
parameters? That trade-off is exactly why it exists.

### 8.9 Model 2 — Fine-tune the trained MobileNetV2 model

Continue from the trained transfer-learning checkpoint and unfreeze the final 30 backbone layers, while keeping BatchNormalization layers frozen.


In [ ]:
mobilenet_ft_model = mobilenet_tl_model
mobilenet_ft_base = mobilenet_ft_model.layers[0]

n_trainable_layers = unfreeze_last_layers(mobilenet_ft_base, 30)

print("Trainable backbone layers:", n_trainable_layers)
print("Fine-tuning from the already-trained transfer-learning checkpoint.")


### 8.10 Reuse the trained classifier head

In [ ]:
mobilenet_ft_model.summary()


### 8.11 Recompile with a smaller learning rate

In [ ]:
mobilenet_ft_model.compile(
    optimizer=keras.optimizers.Adam(1e-5),
    loss="binary_crossentropy",
    metrics=make_metrics(),
)

mobilenet_ft_trainable = count_trainable_params(mobilenet_ft_model)
print("Trainable parameters:", f"{mobilenet_ft_trainable:,}")


### 8.12 Fine-tune the model

In [ ]:
t0 = time.time()

mobilenet_ft_history = mobilenet_ft_model.fit(
    mobilenet_train_ds,
    validation_data=mobilenet_val_ds,
    epochs=EPOCHS_FT,
    callbacks=make_callbacks(),
    verbose=2,
)

mobilenet_ft_train_time = time.time() - t0


### 8.13 Training curves — every tracked metric, per epoch

In [ ]:
plot_all_metrics(mobilenet_ft_history, "MobileNetV2 — Fine-Tuning")


### 8.14 Evaluate on the test set

In [ ]:
mobilenet_ft_y_test, mobilenet_ft_loss, mobilenet_ft_acc, mobilenet_ft_y_pred, mobilenet_ft_y_prob = evaluate_and_report(
    mobilenet_ft_model,
    mobilenet_test_ds,
    "MobileNetV2 — Fine-Tuning",
)


### 8.15 Confusion matrix

In [ ]:
plot_confusion(mobilenet_ft_y_test, mobilenet_ft_y_pred, "MobileNetV2 — Fine-Tuning")


In [ ]:
results["MobileNetV2 (Fine-tune)"] = dict(
    test_loss=mobilenet_ft_loss, test_acc=mobilenet_ft_acc,
    params=mobilenet_ft_trainable, train_time=mobilenet_ft_train_time,
)
histories["MobileNetV2 (Fine-tune)"] = mobilenet_ft_history

**Takeaway.** MobileNetV2 is designed for efficiency. Judge it not only by raw test accuracy, but also by its much smaller parameter count and potential suitability for mobile or edge deployment.


## 9. EfficientNetB0

**EfficientNetB0** (2019) is the newest architecture in this lineup. Rather than hand-designing depth,
width, and input resolution separately (as earlier architectures did), its authors searched for the best
way to scale all three *together*, then used that recipe to produce a whole family of models
(B0 through B7) at different size/accuracy trade-offs. B0 is the smallest member of that family —
still competitive with much larger, older architectures despite its modest size.

### 9.1 Resize and preprocess the data for EfficientNetB0

In [ ]:
IMG_SIZE_EFFNET = (224, 224)

def effnet_train_preprocess(image, label):
    image = tf.image.resize(image, IMG_SIZE_EFFNET)
    image = data_augmentation(image, training=True)
    image = keras.applications.efficientnet.preprocess_input(image)
    return image, label


def effnet_eval_preprocess(image, label):
    image = tf.image.resize(image, IMG_SIZE_EFFNET)
    image = keras.applications.efficientnet.preprocess_input(image)
    return image, label


effnet_train_ds = (
    train_raw_ds
    .map(effnet_train_preprocess, num_parallel_calls=tf.data.AUTOTUNE)
    .shuffle(N_TRAIN, seed=SEED)
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)

effnet_val_ds = (
    val_raw_ds
    .map(effnet_eval_preprocess, num_parallel_calls=tf.data.AUTOTUNE)
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)

effnet_test_ds = (
    test_raw_ds
    .map(effnet_eval_preprocess, num_parallel_calls=tf.data.AUTOTUNE)
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)


### 9.2 Model 1 — Transfer learning: bring in the pretrained EfficientNetB0 backbone and freeze it

In [ ]:
effnet_tl_base = keras.applications.EfficientNetB0(
    include_top=False,
    weights="imagenet",
    input_shape=(*IMG_SIZE_EFFNET, 3),
)
effnet_tl_base.trainable = False

print("Backbone layers:", len(effnet_tl_base.layers))
print("Backbone parameters:", f"{effnet_tl_base.count_params():,}")


### 9.3 Build the classifier head

The pretrained backbone is followed by `GlobalAveragePooling2D`, dropout, and a sigmoid output. Global average pooling keeps the head compact and helps reduce overfitting compared with flattening the full feature map.


In [ ]:
effnet_tl_model = keras.Sequential(
    [
        effnet_tl_base,
        layers.GlobalAveragePooling2D(),
        layers.Dropout(0.30),
        layers.Dense(1, activation="sigmoid"),
    ],
    name="efficientnetb0_transfer",
)

effnet_tl_model.summary()


### 9.4 Compile — only the head's weights are trainable

In [ ]:
effnet_tl_model.compile(
    optimizer=keras.optimizers.Adam(1e-3),
    loss="binary_crossentropy",
    metrics=make_metrics(),
)

effnet_tl_trainable = count_trainable_params(effnet_tl_model)
print("Trainable parameters:", f"{effnet_tl_trainable:,}")


### 9.5 Train Model 1 (transfer learning)

In [ ]:
t0 = time.time()

effnet_tl_history = effnet_tl_model.fit(
    effnet_train_ds,
    validation_data=effnet_val_ds,
    epochs=EPOCHS_TL,
    callbacks=make_callbacks(),
    verbose=2,
)

effnet_tl_train_time = time.time() - t0


### 9.6 Training curves — every tracked metric, per epoch

In [ ]:
plot_all_metrics(effnet_tl_history, "EfficientNetB0 — Transfer Learning")


### 9.7 Evaluate on the test set

In [ ]:
effnet_tl_y_test, effnet_tl_loss, effnet_tl_acc, effnet_tl_y_pred, effnet_tl_y_prob = evaluate_and_report(
    effnet_tl_model,
    effnet_test_ds,
    "EfficientNetB0 — Transfer Learning",
)


### 9.8 Confusion matrix

In [ ]:
plot_confusion(effnet_tl_y_test, effnet_tl_y_pred, "EfficientNetB0 — Transfer Learning")


In [ ]:
results["EfficientNetB0 (Transfer)"] = dict(
    test_loss=effnet_tl_loss, test_acc=effnet_tl_acc,
    params=effnet_tl_trainable, train_time=effnet_tl_train_time,
)
histories["EfficientNetB0 (Transfer)"] = effnet_tl_history

**Takeaway.** EfficientNetB0 is from 2019, four to five years newer than the other architectures here —
see whether "newer" translates into "better transfer learning result" on this particular task, or
whether the gap has more to do with parameter count and depth than release date.

### 9.9 Model 2 — Fine-tune the trained EfficientNetB0 model

Continue from the trained transfer-learning checkpoint and unfreeze the final 45 backbone layers. BatchNormalization layers remain frozen.


In [ ]:
effnet_ft_model = effnet_tl_model
effnet_ft_base = effnet_ft_model.layers[0]

n_trainable_layers = unfreeze_last_layers(effnet_ft_base, 45)

print("Trainable backbone layers:", n_trainable_layers)
print("Fine-tuning from the already-trained transfer-learning checkpoint.")


### 9.10 Reuse the trained classifier head

In [ ]:
effnet_ft_model.summary()


### 9.11 Recompile with a smaller learning rate

In [ ]:
effnet_ft_model.compile(
    optimizer=keras.optimizers.Adam(1e-5),
    loss="binary_crossentropy",
    metrics=make_metrics(),
)

effnet_ft_trainable = count_trainable_params(effnet_ft_model)
print("Trainable parameters:", f"{effnet_ft_trainable:,}")


### 9.12 Fine-tune the model

In [ ]:
t0 = time.time()

effnet_ft_history = effnet_ft_model.fit(
    effnet_train_ds,
    validation_data=effnet_val_ds,
    epochs=EPOCHS_FT,
    callbacks=make_callbacks(),
    verbose=2,
)

effnet_ft_train_time = time.time() - t0


### 9.13 Training curves — every tracked metric, per epoch

In [ ]:
plot_all_metrics(effnet_ft_history, "EfficientNetB0 — Fine-Tuning")


### 9.14 Evaluate on the test set

In [ ]:
effnet_ft_y_test, effnet_ft_loss, effnet_ft_acc, effnet_ft_y_pred, effnet_ft_y_prob = evaluate_and_report(
    effnet_ft_model,
    effnet_test_ds,
    "EfficientNetB0 — Fine-Tuning",
)


### 9.15 Confusion matrix

In [ ]:
plot_confusion(effnet_ft_y_test, effnet_ft_y_pred, "EfficientNetB0 — Fine-Tuning")


In [ ]:
results["EfficientNetB0 (Fine-tune)"] = dict(
    test_loss=effnet_ft_loss, test_acc=effnet_ft_acc,
    params=effnet_ft_trainable, train_time=effnet_ft_train_time,
)
histories["EfficientNetB0 (Fine-tune)"] = effnet_ft_history

**Takeaway.** EfficientNetB0 is relatively sensitive to fine-tuning settings. The low learning rate, frozen BatchNormalization layers, early stopping, and learning-rate reduction are intended to make adaptation more stable on this small dataset.


## 10. Compare All Five Backbones at Both Training Stages

Each architecture contributes two result snapshots:

- the frozen-backbone transfer-learning checkpoint;
- the same model after controlled fine-tuning.

The goal is not simply to find the highest accuracy, but to compare **generalization, efficiency, trainable parameters, and training cost**.


In [ ]:
comparison = pd.DataFrame(results).T
comparison.index.name = "model"

comparison["test_acc"] = comparison["test_acc"].round(4)
comparison["test_loss"] = comparison["test_loss"].round(4)
comparison["params"] = comparison["params"].astype(int)
comparison["train_time"] = comparison["train_time"].round(1)

comparison = comparison[
    ["test_acc", "test_loss", "params", "train_time"]
]

comparison.columns = [
    "test accuracy",
    "test loss",
    "# trainable params",
    "train time (s)",
]

comparison.sort_values("test accuracy", ascending=False)


In [ ]:
arch_names = [
    "VGG16",
    "ResNet50",
    "InceptionV3",
    "MobileNetV2",
    "EfficientNetB0",
]

transfer_acc = [
    comparison.loc[f"{name} (Transfer)", "test accuracy"]
    for name in arch_names
]

finetune_acc = [
    comparison.loc[f"{name} (Fine-tune)", "test accuracy"]
    for name in arch_names
]

x = np.arange(len(arch_names))
width = 0.35

plt.figure(figsize=(11, 5))

bars1 = plt.bar(
    x - width / 2,
    transfer_acc,
    width,
    label="Frozen backbone",
)

bars2 = plt.bar(
    x + width / 2,
    finetune_acc,
    width,
    label="Fine-tuned",
)

for bars in (bars1, bars2):
    for bar in bars:
        height = bar.get_height()
        plt.text(
            bar.get_x() + bar.get_width() / 2,
            height + 0.005,
            f"{height:.1%}",
            ha="center",
            fontsize=9,
            fontweight="bold",
        )

plt.xticks(x, arch_names)
plt.ylim(0, 1.05)
plt.ylabel("test accuracy")
plt.title("Transfer learning vs. fine-tuning by architecture")
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
plt.figure(figsize=(11, 6))

for name, history in histories.items():
    plt.plot(
        history.history["val_accuracy"],
        label=name,
        lw=2,
    )

plt.title("Validation accuracy across all training stages")
plt.xlabel("epoch")
plt.ylabel("validation accuracy")
plt.legend(bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=9)
plt.tight_layout()
plt.show()


### Portfolio interpretation

After running the notebook top-to-bottom, use the final table and plots to answer four practical questions:

1. **Which backbone generalizes best?** Compare held-out test accuracy, loss, ROC-AUC, and the confusion matrix.
2. **Did fine-tuning actually help?** A fine-tuned model is only better if validation/test performance improves without clear overfitting.
3. **Which model is most efficient?** Compare accuracy against trainable parameters and training time. MobileNetV2 and EfficientNetB0 are especially relevant here.
4. **Which model would you ship?** The answer depends on the deployment target: maximum accuracy, fast inference, small model size, or low training cost.

### Reproducibility checklist

Before publishing final numbers:

- restart the runtime;
- use a GPU runtime if available;
- run **all cells in order**;
- confirm there are no warnings or failed cells;
- make sure the final comparison contains all 10 stage results;
- commit the executed notebook only after the full run completes.

This notebook is designed as an end-to-end portfolio project demonstrating **transfer learning, controlled fine-tuning, regularization, evaluation, and architecture comparison**.
